[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module2_DataSimilarity/02_NLP_TFIDF.ipynb#copy=true)


# Introduction to Natural Language Processing: TF–IDF

## Learning objectives

By the end of this lesson, you should be able to:

- Describe how a collection of documents becomes a feature matrix.
- Compare bag-of-words counts with TF–IDF features.
- Interpret high-weight terms and identify limits of this representation.

**Student Learning Outcome (SLO 3):**
> Represent text as a numerical feature matrix and explain what that representation retains and loses.


## Part 1 — Build a bag-of-words matrix by hand

Start with two short **documents**: A = `cats chase mice` and B = `dogs chase cats`. First choose a **vocabulary**, an ordered list of all words we will count: `[cats, chase, dogs, mice]`.

For each document, write a 1 when a vocabulary word occurs and 0 when it does not. Document A becomes $[1,1,0,1]$; B becomes $[1,1,1,0]$. Stacking the rows creates a **document–term matrix**: rows are documents, columns are vocabulary terms, and entries are counts. This is the bag-of-words model—it keeps which words occur and how often, but not word order, grammar, or most meaning.


In [ ]:
tiny_documents = ["cats chase mice", "dogs chase cats"]
tiny_vectorizer = CountVectorizer()
tiny_counts = tiny_vectorizer.fit_transform(tiny_documents)
pd.DataFrame(tiny_counts.toarray(), index=["A", "B"],
             columns=tiny_vectorizer.get_feature_names_out())


### Pair practice — predict before running

Write the document–term row for `mice chase mice`. Would `cats chase mice` and `mice chase cats` have identical bag-of-words vectors? What limitation does that reveal?

> **YOUR ANSWER:**


## Part 2 — Build a document–term matrix with code

The hand-built table is small enough to inspect, but real collections can contain thousands of documents and terms. `CountVectorizer` automates the same three steps:

1. split each document into tokens (words);
2. create a shared vocabulary from the tokens;
3. count each vocabulary term in each document.

The code below uses four slightly longer documents. It removes common English stop words such as `the` and `and` so that the table emphasizes potentially informative words. Read the resulting table as you read the hand-built one: each row is a document and each column is a term.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

documents = [
    "The movie has a thoughtful plot and excellent acting.",
    "Excellent visuals but the plot is slow.",
    "The service was slow and the food was cold.",
    "Warm service and thoughtful food made dinner excellent.",
]

count_vectorizer = CountVectorizer(stop_words="english")
counts = count_vectorizer.fit_transform(documents)
pd.DataFrame(counts.toarray(), columns=count_vectorizer.get_feature_names_out())


## Part 3 — Why raw counts are not always enough

Raw counts treat every occurrence equally. But a word found in nearly every document, such as `excellent` in this toy corpus, may be less useful for telling documents apart than a word found in only one. TF–IDF changes the weights so that locally frequent but globally rare terms receive more emphasis.


## Part 4 — TF–IDF: downweight common words

Term frequency measures local importance in one document; inverse document frequency downweights terms that occur throughout the corpus. In a simplified form,

$$\operatorname{tfidf}(t,d)=\operatorname{tf}(t,d)\log\left(\frac{N}{df(t)}\right).$$

Here $N$ is the number of documents and $df(t)$ counts the documents containing $t$. Library implementations may apply smoothing and normalization, so their numerical values can differ from a hand calculation. The name **TF–IDF** combines term frequency and inverse document frequency.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf = tfidf_vectorizer.fit_transform(documents)
tfidf_frame = pd.DataFrame(tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_frame.round(2)


### ✏️ Written response 1

Choose one document and name its two most informative terms according to TF–IDF. Why are raw word counts alone less useful for comparing the documents?

> **YOUR ANSWER:**


## Part 5 — Preprocessing is a modeling choice

Lowercasing merges `Excellent` and `excellent`. Stop-word removal can save space, but removing `not` can harm sentiment analysis. Bigrams preserve short phrases such as `not good`, while producing a larger, sparser vocabulary.


In [ ]:
bigram_vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=None)
bigram_matrix = bigram_vectorizer.fit_transform(documents)
terms = bigram_vectorizer.get_feature_names_out()
print("Example terms:", terms[:18])
print("Matrix shape:", bigram_matrix.shape)


### Practice

Add two documents about a topic of your choice. Compare the vocabulary and matrix shape with and without bigrams. Which preprocessing choice would you document for another researcher, and why?

> **YOUR ANSWER:**


## Vocabulary checkpoint

- **Corpus:** the full document collection.
- **Vocabulary:** the chosen unique terms/features.
- **Token:** a unit, often a word or short phrase.
- **Sparse matrix:** a matrix containing mostly zeros.

Which of these changes when a new document has a new word? Which changes when an old word is repeated?

> **YOUR ANSWER:**


## Part 6 — Matrix shape and sparsity

The matrix has one row per document and one column per vocabulary term. Adding bigrams expands the number of possible columns faster than it expands the number of nonzero entries, so text matrices are usually sparse. Sparse storage lets us work with large corpora without recording every zero.


### Feature-design challenge

Would you use character n-grams, word unigrams, or word bigrams for detecting misspellings in product reviews? Make a choice, name one benefit, and name one tradeoff.

> **YOUR ANSWER:**


## Limits of bag of words

Bag-of-words models treat `dog bites person` and `person bites dog` as nearly identical. They also struggle with long-range context and sarcasm. These limitations motivate embeddings and transformer models, but TF–IDF remains a transparent, fast baseline—and a useful feature representation for this course.


## Key ideas

TF–IDF produces a sparse feature matrix that can be used with many supervised and unsupervised methods. It does not understand context, sarcasm, negation, or the meaning of unseen words. In the next lesson, we use these features for a supervised text-classification task.
